# Projeto II - Regressao de Iluminacao Residencial

Notebook para o projeto `Estimativa Reacional Regressiva Perceptiva Aplicada Sobre Iluminacao Residencial Dinamica`.

Este notebook prioriza o CSV real coletado no Wokwi. Se ele nao existir, cai automaticamente para o dataset sintetico.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACTS_DIR = PROJECT_ROOT / 'training' / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
ADC_MAX = 4095.0
REAL_DATASET = ARTIFACTS_DIR / 'wokwi_training_dataset.csv'

## Carregamento do dataset

Se `training/artifacts/wokwi_training_dataset.csv` existir, ele sera usado. Caso contrario, o notebook gera um dataset sintetico.

In [ ]:
if REAL_DATASET.exists():
    df = pd.read_csv(REAL_DATASET)
    source_name = 'CSV real do Wokwi'
else:
    lux = np.geomspace(0.2, 10000.0, 500)
    adc_raw = np.clip(np.round(ADC_MAX * lux / (lux + 35.0)), 1, ADC_MAX - 1).astype(int)
    df = pd.DataFrame({'adc_raw': adc_raw, 'lux_referencia': lux})
    df['percentual_adc'] = df['adc_raw'] * 100.0 / ADC_MAX
    df['lux_estimado'] = np.nan
    df['ratio_feature'] = df['adc_raw'] / (ADC_MAX - df['adc_raw'])
    df['log_lux_referencia'] = np.log1p(df['lux_referencia'])
    source_name = 'dataset sintetico'

print('Fonte:', source_name)
df.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(df['lux_referencia'], df['adc_raw'], s=18)
plt.xscale('log')
plt.xlabel('Lux de referencia')
plt.ylabel('ADC raw')
plt.title('Lux x ADC')
plt.grid(True, which='both', alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(df['adc_raw'], df['ratio_feature'], s=18)
plt.xlabel('ADC raw')
plt.ylabel('ratio_feature')
plt.title('Feature derivada do divisor')
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Treino da regressao

In [ ]:
feature = np.log((ADC_MAX - df['adc_raw']) / np.maximum(df['adc_raw'], 1.0))
X = pd.DataFrame({'log_inverse_ratio': feature})
y = np.log(np.maximum(df['lux_referencia'], 1e-9))
y_real = df['lux_referencia']

model = LinearRegression()
model.fit(X, y)
pred = np.exp(model.predict(X))

rmse = root_mean_squared_error(y_real, pred)
mape = mean_absolute_percentage_error(y_real, pred) * 100

print('peso   =', model.coef_[0])
print('bias   =', model.intercept_)
print('rmse   =', rmse)
print('mape % =', mape)

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_real, pred, s=20)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Lux real')
plt.ylabel('Lux previsto')
plt.title('Ajuste do modelo')
plt.grid(True, which='both', alpha=0.3)

## Exportacao para o firmware

In [ ]:
artifacts = {
    'weight': float(model.coef_[0]),
    'bias': float(model.intercept_),
    'rmse': float(rmse),
    'mape_percent': float(mape),
    'samples': int(len(df)),
    'source': source_name,
}

json_path = ARTIFACTS_DIR / 'projeto_ii_wokwi_model.json'
json_path.write_text(json.dumps(artifacts, indent=2), encoding='utf-8')

header_path = PROJECT_ROOT / 'main' / 'model_params.h'
header_path.write_text(
    '#ifndef MODEL_PARAMS_H\n'
    '#define MODEL_PARAMS_H\n\n'
    '// Projeto II: regressao logaritmica sobre a curva real do LDR no Wokwi.\n'
    '// Arquivo gerado pelo notebook.\n'
    f'#define PROJ2_MODEL_ADC_MAX {ADC_MAX:.1f}f\n'
    f'#define PROJ2_MODEL_WEIGHT {artifacts["weight"]:.6f}f\n'
    f'#define PROJ2_MODEL_BIAS ({artifacts["bias"]:.6f}f)\n\n'
    '#endif  // MODEL_PARAMS_H\n',
    encoding='utf-8'
)

print('json   ->', json_path)
print('header ->', header_path)